---
title: "6.2 Internet use and urbanization in 2019 · Complete solution"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/projects/02-internet-urban/02-solution.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/projects/02-internet-urban/02-solution.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If the GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. Each notebook starts afresh.

**Time:** 60–90 minutes, split across sessions. **Before you begin:** Complete Parts 0–5. Attempt the [starter notebook](01-starter.ipynb) before reading the worked analysis.

## You will learn

- Define and audit a complete-case sample from new real public data.
- Produce a labelled chart and a comparable pair of formula regressions.
- Export a table with source, uncertainty and interpretation notes.

## Why it matters

Do more urbanized economies also report more internet use in 2019? This second project provides less step-by-step guidance: specify the sample and units, choose a readable plot, compare simple and region-adjusted associations, export results and explain limitations. The two percentages describe different quantities, so their relationship is not an accounting identity. Start with made-up numbers, then use real public observations; attempt the separate starter first.

## Worked example

### Prepare the analysis libraries

Run this prepared cell before importing analysis packages. It uses the course dependency pins and installs only if required. If it asks for a restart, restart the notebook session and run from the top. Installation needs internet; no paid software is needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. These packages run inside the notebook session; this does not install Python on your laptop.

### Establish the units with a tiny fictional example

Import the existing course tools and inspect these made-up numbers. They demonstrate columns and units, not a real empirical pattern.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
plt.rcParams.update({"font.size": 12})
practice = pd.DataFrame({"economy": ["Northland", "Eastria", "Southport"], "urban_pct": [20., 50., 80.], "internet_pct": [10., 45., 75.]})
percentage_point_difference = 60.0 - 40.0
print(practice)
print("40% to 60% is a difference in percentage points:", percentage_point_difference)

A change from 40% to 60% is 20 percentage points, or a 50% increase relative to 40%. The regression below uses percentages on their 0–100 scale, so interpret a slope as percentage points per percentage point.

### Prepare the pinned public-data snapshot

Run this prepared setup. It uses the course files locally or recreates the exact frozen public-data snapshot bundled in this notebook. It verifies each checksum. The source observations are from 2019; the snapshot was retrieved on 2026-11-11. No account, key or private records are needed.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared project-data setup"
from pathlib import Path
import hashlib
import base64
import zlib

# Frozen public-data snapshots, generated from the rights-documented course inputs.
expected_files = {'internet_urban_2019.csv': '95a88e1697502cff165ac757b29e8bac5c361c97615e4701b1dbfdd4d6a839e9'}
saved_snapshots = {'internet_urban_2019.csv': 'eNqlm1tzG8mRhd/7VyD0sPZGlBFd96pH8M4hAXIAUhrpxdEke8i2QLQMApKpX79fNjSzXtvo1o6oiRmKQ2RXZWWePCcru75vV+3z61/v24da1bu/qHX92LQr9VpXa9WsNvV6VW/++ul+o7bru2ol3xWTg3dqst7eVeqy2jSr0eS5Xjf31ei/RofVurm7q6uVMqXOSgUzNimZGEwKKflicnKqJr8+PlWr5mXDb72ZNg8Py3p0XL1s1GjWrjdPo8mvYkyN/un3sHxdfey+fbOzrOM4lPJlkzJ+rJ2NxqdkcyptMTm9UpPVY7us1GJ795dF9VStMfLNcPd5a8baZH7dehXCOAQbk7E6ld7y+csDNVmy26ZSx9t1+6mWrdWrzbpajiYvzTcbIY29L23W2vDBMigfxjHYXEZtEsthu7MjFvLQrte9hnI5jjqFHHxWCaOYjNmGEIzDxvxY3a6aTf0wmqyru9Hxc7OuNvXLH/ddzvgr5xyzD9o4q5IflyYnE6xNSWd5Jse0fmShzWrwkGMeZxeyN6V10Xutsh4nH2JyzmubSoO5Keae6yF/hrF3NrscUggaXwTWVbpYWutj6UwqJgsM7daxGi2q5xZ77L6z0e3zvvm1uf8We8mOS++MjjgzmuiKyQ27YkuP22pUrR5GB9X6bvswvD89LrPxuCso48axTNZGb1wph3O7UJPti2yk6VlLtkRrzqUWZ8ex5WfBxZSSKwM2bnY2+r3D53iqwR8pEmxd+FkdQ+nwesn/KSYfjtXka02aNn9j8X22ZEvyVQavvYvKp7G2JWevc+QrFwdH5+pgu96uHpq9ORTGZTQ6Ezha6TSOKeiYMKdTLA6OL9VBvXxsts8DkW8ih5RKcshp2WSZgw5W65QNezo4nmFn1az2rsKUY1/KbpxXjlVoEq8sfS6NS8XByUS28ZEwHp1UL+1eK5l1kMRgSvbAACcdxqyBLHau1Nqzo9MjdVCtHpfVQ/3ypBbtVvLt953YcmyT1TxZy/fJGMAoWMdZs4vTOctYPlYDZxxil0qpzE5O1qpo2VAw1hjczNEXB2dYqp7WVbP6IQCIpfalIzs57aRYNpYX6g2mq+fqRY1unuo3Q4nB4YVsAQ7rJKKS8dEFy74J0uLg/EwdtC+kfJdrZ/X6a/3YfhY86XNAhyXgCBEQTQjK5bELrNSB9DF6TvRyLoFVrbcvvRFuiMesAaWsSxJPxTh2BEbklD0hhycvP4ih5ms9tE+SIwYsEQXKmTH4Tj0zmkphbHEwvcXM+lmA5NsR7Ox8Q6HOtVckQ7tsPjeDYONYZnSSiCREyEY8Eq11OQUBHsmsOSG9rr42y0HgsuMs+J6zBkETTqHQgYX8vUzO5ICtAyUoWD20L4PWCPAylpw2HiFdktMa+HLeJTFEmoIWdTM6kqN5qZbVcw8eehVB0mwsiR6TDRFH3mDiaSus4F9zi9+Vgpg5f2UzqOMCxdqRao5Hv8Mf7eblS7XaX+4BBjKSnYckZcaS7CG7xOKDKw4nJ+r3CPp1V1zm9aft3ZL17rMYQZ1sICBUBacJNiqVLV3JYRksztQh6/nPMUF95FwAqGQlb6INOECAnawvDs+O1eJLswHHlyROL3ha6ngoIQ7eeh2jSvAInEqAAhhABsbO1SE4sKqXo/MXsdebNMKKcCwIrksTUgwYuMRAsxzMEaEQAlmaVIPCaMOuIAEYswkrM7Gy6qmQwY3LVMI+DKesvRBHYo0E1kFrMKU4PH+rDttNPXr40/nntlnX+6ldEM6ZvfPBk7VeeT3GP7iX2kY+4OPpnONhK2276jMjRAzctcqTSS7FTInSmZgtDq+O1JvDFo6pRkf181jiZfxmry1thWBRtk0SBEkRUpP5w3/x8dXp77YGzEhNgmCyDxXsmDOHtkLhoG0eMxxVu2yf74ZRpiNWIZcwVQ/fsjt0zLgreZ0AWqxNsfbcrgGGnuVQ8tKu5BHHkF4vjCaBOcXhNedV3bWjt/X6Yf9hBXLBWVI7+gwBScK2ghFG4AlsLyuZE8Mt1Ws0l08NxaEemxKyFpyz3sCS2BmJ6RN8y/JjFnZ7oA6/Q7tILY4Sk8ELSTKdJfZoKHP8uCQkb99haV3dV+2gECLnbYR1a4F2W3Jc73Fw9fqMP37LzCEbUkoO31+rw9dPQ9UPagaoQBoCPGoncWKIUXh05LhYO1zx8Gt9/zRAOmEyAaR0TsNmDE6QuhqDM5Qj5IMtjo5v1Sn1r1q99hoSgokeIOjIKR3koHCD1KaQMiKqOPrpXB39rbkD+5s/Tm2o1eQUuKwlkiJcFIj0YAF7PppO1FH73Ky+I46onTYZIR1dSMKUIHQOECCyiqPZhTqqV8/V+mMvPidCLzhRRbAF1zkBPhJchmmXuLU4Ist+W9E/1ZxhwRXFXpZ0AQYcwis4SgjQVBx9mKBdH2thmn/ci57iDEB0xMmMHRmNVCTVdfTF8eGtOr7fwhnWg8wJniBIVQYKLknOWVAOjk/fqzfHj6+fWFQnaXeo98dXGwF8Q5xaWTDcwsmSNTggxTggA47BkON1s1nX+ymCBjmMlSIBmvkxlQtctaBZEgOLa7X4JKR7WMYHbw3QE+WcoLIZhIZvwZ2wcqOOXzbtgA4WUWSQP0AFZG3Hij1ZhwNDkEgqjm/O1PHmqWk/NT0b8uOcnHRJojJszgmF9EJUYAYn5zN10qyGKEZiC7AJqoIU5pi7/HdWxBqYarUrTkjbk+ZvTU9pt2NEeMpIha5DAoWUSgOMFidw2RPWfV/3LoJcZO9oAJybKMcxjQU0ovA30C1g50qdVOu2/j6Wg3DOHTsQTog7TxZTib77dbuq5ddGJ/XDeLTYvBCVe7dlpT5YdJRLyhgYRiZJUFPkdnE6OVCnFL/97ILM8L/zJURPiej0pBdI6IrTg/lvLZ+LZvX40PZraIMpAiPhDLIsifYBXLIzcnJUm+L0+AqMbtePg/0s4gUT5HrIAB9gGoTTZo4PjMHQ2USdPvURbTwh7LNDEONIe5FfcLji9ByXNHc8cVOte49Hytwp4Xm6bVY9+WrsmBgABbUWrqaNtB44zSAi6XR6oN6cVh0X2mnZvTRPi86nUnKMkBHg2og6yoR9cTo7+LaKvxw0aJrt/lwjGIw01QzoY4XgecRwJm2iwczP6vjv22rTAsrL0cC+ULzINeIKL6K5sAAfh80KvBen80N1uq7r/oSJHp0H/hEFyGhUKwmjE+QajUIGgeGn8yOx0ymUIR7kxuQ7stH7zs9drfHoSJyPnctuPYNA0nV2OhgQEQ44F6c3U5xbbepntOKgLHZjR64gg8GikvBysiEDyhpRtCzktrPWozlVRmJEaaMYzKAm+cx7PvNaDbc4iWZ8EHh61xhyXavZwadKn4qzC/j7Gfx9dCH/WkzmatSpnR7wyMJNrDQ1pEmRYhf0Z7MjhZ0HKOUgGUQGW9YAsRDYIAVAVoqX7sq01InibA4BX7eY6U36KHo6dxrcc8SuM1Rag1wC4Evs3Jyrs6qBlA0tiZQPTkQb52N2AsPn7DHli7PbmTrbrh6r9esA37QizW0ofQntkAwQpDZR1GAoc3F+NFPnq4dvSL3XwS6OeTy1n60haHwHSMLoSEpsu1icTzH0At9ofx1N+9ulsh0JXf7IZmIqzjkqViHV9196FQZghzuQaU5ZkADawweS8FCeSbqcr+vBqkuu8EWRJNLgTF0JlVoDlAIGRgzN1JtzoEN1Fe+5uf9REiU0zyNJYc8R5PACIobwTNJ4cR7Nez7/mcVXf//jzyCL4WNWCLm0VDtik0KiuHvSOjpTnC9w0P2ggzJO5iRR44QKzEhaIVlKr4PxkwEsdjHndNdVvfzjy0VFYTtFCFwi0a1kbeAAIFD8Q1AW5zcTdb6plq8DTV1yNGrEf3SCIELnCG/wLCT8YIufJlP1U/VcfY8uQZA5SAP1GdnsuzunQITxD14sQcKfrubqp3b98CO3azxFfFmyWdk08jcHZIvhAcVP1zMW+0kyZi+6QRLBEFJPUwcpiQJ3OgghIYU0Ri4mH9RF9bX6+NRdA/ZfVcgJSHsMOpGcMCfYNHmIBqUshuLieKYu6tVrLymRsoca4CR17LoVWioRmzIavlNcnLKe1/Xj69f/VWF9R0r5UdaNE3Fh4NQ+ke7FxZlI+ue79qEPmcASKRuR6uUoRILkArSGvAaZDPs5n6uLhlMnEnoATkv/NMM4osSAEeaGyJJ2bC4uZhO12Ixhj5vNS9d/n9Wfm8GiQmBSlBAdxBUucnKvqZPNqBFcRFy9uWhFQn3Dmv3nD03AKZSj5IzdNWWyhLv1wi1Z37sbdbH9QlX5oUsML22IINqeTXcl9HJyxSbb0fXRvLc2lN2NSumkaGXp6qQSoMNScXkwU5f1XbVqfyB/2HAnQCFvwu677r30o7QvLU+Yq8vmrlPoe/kkwChZnYOTgElkUYDtOGrI5cF7+fjrD8h7aWgIrUfWSa0hEuWKy5TZAsGXh7vYudzef0cjUUvbFgbiu7tRslKAKRDQCUvnx6y0vn/a1KuXTT0gnrtmNiCZtdJOVKdoZrnIk0O5YE3rZnRZrT7+e8m1kA0Wj8614jgnN0j4iqiwuGtBRNQv7eZp/+Wfk5TUUfCUdLIiEIP4B9ABc3wuLm9u2cnmaTs0EsDBe1KGmiZUSCp3HIsstCg01CrLuf1FXW7/UYMS2/Vjr0PiWCi8rMmaaMRB0E65folyP4aptxM5ns8DKxLKLr1wR2pb0XjIAmtsQB8CntPJIXEkLczvI66YA2iy9ECDsOAu66aTky5kptVaouXPJyiL+6fRJ/7639/V1Zzy6Gm7bu/v2x9IOicOs6JYQQVEAsGZ5E4GwYCgCr6YHl7xmFXFU3rbdrs1HU345eVD+7mfP1vhNBpA0crJ/UvXuum68pg4VVMk1mP1co/o3X/3DHMXVuFtd43M+nM00D25R50evcXG8qH5XL/8+8VYHMvvsukoD4cS8WQHmTek3/T4FzWt/9Hct9+B+0FkL5+WepZFL1rvM96zwCSmzi5ZxPrlqVr+00XSXnzN4yjqN0v7PYwJNCNq1HWtq+nF0bc7MaKufmiHE4oy1I0QUYlAw1COu2Ei8BU9aVnb5bk4aP+YAriEcgBLnFdGNmbk2kPUOXJienkjn978AJomlEVMcrcQZfpIi9DMOK6UNiBimT1PCW905rN0Pvb2k2T6RWghmyQKoErGSZaiJtjk7FgiFyCtH9ftQDyCDiGwwZgCYBr8GOxB3Um9g8hh61RsPba9Iyted2qZuExGJhGExaLCCAfIuwfJprPr3UHW65VkfoOOHo6NHYeWZhWHB1dna1cfWM5X6dX8fbv/okijTzrCI20I9qhZC9uRjhk25nKK23Wz6QB6byT4cSldzGyFMlFapJsa5VoPILxdfDPRbF/6rqu+uQBJhJAR9gYrziReQu96MOZdF47Vl6av0wwBDHJh3F3pWaqFC/AiJEExfb/oPv/aK28F65KwvVLiJGm5Ipd7u1jCcik8ppihKWYIw7umt9Uk926Uvu6GnMdnGQgxqJRczA4v1az+AlYsf0vTvYdKcEhGBdGogj7F7HiuZs1jve5tt0vHEpYjTkAncwqCGVDN2elk9+m+s2Ttzsm8TTe9lIPW1F0nCBaK2fkhBsDc6nFbfUcrBW5KVDgtliQJA651uIKNXAJXtQT5YF9ZCnQCB0rhDnL1yA9Skn4TPwWVcSkUmpT5Ur0OXRvJl0MT4xQj/Vxpcye5gpbDLmbXcjSfquW/UyFplYdQahMk8bNcGhgZdJAJptn8lpBoawGQfUcpYxlWmvXfpoJmH3ZR8KGudqJ8b0SKlJcWBWo8sXFqETQQ1QdTuZrO1NXzjwhSuRLx2XZnFLvbkFLuopykcHE9uVC//f4PTZTKeAtmtVxKSn+/a11bLdemIfOYGY9ZodMH66mVGuW0dAjJ0Q6DozfwbdtVieKa9Liu19vvuD/L0iqwhFSWCVM8C9Wglln8rEtfXFOZr5+aZfPpU7Oq+4qyjCAAMkJOS0PCexkt81K4iA92XVxfvmN/y2rbk+hRBn9Ki65HL1APimuKyXX1aVuNJEq+tbb3fl6LY5zMVET5Hr5BcSGNKXOYumIr7WBvrESjSpKlKFXNdj4KMLBu8kAu+K/n5+p6W683rUwMtKM/3y4GWShEKsIubLYiO3zHthEe0suULy9WL36Xv93Ex3XdflrWf3oZEsMyygIBx4p4X5qH11SrawJz+0gG97fxKTGguhRhGJRUH62F41FygpY1vcf5Hci9DsZk6i6cCEvZkBApJ9eDEpVdF4GgXByrdzW7OEBldU2D0+pr9UNNJDQrUjB2uZqEKAHzQcutvA3A9PX7E/VNK3DwrwMt3a74QoqARGDel6n4eXKjfq7kPulHugiBSKZilsBX2g1Hy4BCDl0fNBbzq1s1b5+HlB+8q+NcXlAJBdzJ9my1RJNcvZpiDsOYb1/4xEouGOs1B9auBoYYpUFFbAfkVkiCzrjUSM4SnCYU83cTNf/CWfUUSpl+EibXXVPm2M12ge+WJS0mt2pRbR+a7h7+RyYGcnffF0U1eODBCFIFm2X8wUiepmJxNGON0pbct1LpqLkI00+eY0aiFotjPiOcV0rdPhJjRT6F7t7eyfioiBaZPwN9i8XptVo0q8fqE5nbU7wgENLeZKm+JEm60re4PKC+LtvndjXMa42MHKQkDQtlogw1a4IH5RnZxeUxi6jX62p0Wberuu/GPmpJF9P1JGIJqDkHPYXoYOWtOl6OFtXy83eNXoCTmUgGFaNNMinCCrWDROhS7p1ysUCVLKodd1+139HB7l62yHE3Ot612o1mXVdT9WZBgix/vznvn2KT5rBmCaW2ojK6FqyMJVnAoljMcXq9vhtINnSvlnkmCLwQrK4PxQGSaDIKCbQsFkffyFF/zMlErN7RLekdUVekMagTeIuHboieqh3dtM91B4jXeOq++bT/BJ2Dp0RKtXHdAJmWi2WLrwTuFrc4HIUBhRicp6RoyGsP1Pwow1slUcH+rO5mxxZvL9Ri2X6uPn5XtzpJA1FYoe9uYJ2Ufnm7xCNCggOgMTjrDA69odHRDxAN+kXd1lFu1TqBGmUwDjpRLN7J+Cro1t/sc8JopSMFxzacn7zcUCZqU9lhO65/90Edv3wRJzV9qY/Qlns5KyrfygijXLojaojLX6aS+hviu1ojnEd/Ptpu/l9dqcX7Q2Lx9f6pXi7r/ZowBhnz4VtSn/MnkrIMTcjNu8MGZ/4q4vj3Uafdef1hpJVKAzwiBeBORH8obg4n6ma7/rjr9B9WkJ6X7x4qzNQEBCNSzMqEm8fakcwMP/RMPsstirz24bueL9lcgvwyf2WLm9MrddM+tn2vawDxtmtzOD120hwFplDVPPqMjTxVTb/SkNe1vLBIQhdWnSQIs5CrIJc88rrFzU8X6qb6W/NNEvQEotwMUfe7wVx548NKE9k7GclmLxfTzq/y4tKQoe5SwclUkUHK487i5nKhbprndv2XS0hVT/ExovdkTgvaaYR9cLZy9etJ1uLmaoY/KWC9nRovo56U9G4QCbmpPT8J0Jfi5obzAHKah+qhC4+b9q56HOwIRiGHkA4Opht+tnKPg1s4uVzc3LKmLT75EcIgp9jVke7qy7lu+kj63fIaSSB7eMy8c3/zWg8QrxxZKtELAOlu/DxIhMows8x7Y+gthj5Xy20/uTTUSviwgW7BPW4+EIzV6mtvR0mHsZdBbvhjp71l1NPLDDX/LW5PJ+r2sZecaeknCTuT+xOqrJW5Ky29iNIWtxdzdftRXvXpd0A5ZpcwBWlMmWClnQvpjEQhmEo4F7fohNv19ntkgszaeXm1JcJ/vUtC7NgViJZLjodjuV1MfhtOW2y6FxH/wysOcunFwSYSIcohiGwjucSQ3AFmV9x+OFC3X+/q4RyVF0+6SGFDqOkk1MZnsxsHkl558fbwprt/eEt15uNdnG+e6tFuwqnTxcNCm6izu1d8EP4oSoAgZ3xhirew0DdvoaFft/VSbj8PBl+NgohKKxeWZMQXaPayezUsZuLx7emBOpBG48sTS14/Nt8/Ai6XxzL4GBFBVOFQvD2fq/9rA707XoyHKxwHW8rQhWh5TgYvzqaYqjejWd8YVeiyy0MASpAbQmhFk9is5eJYBt6Kt7c36m212labvnwzdgwdkXG0LOzPF+8W1OuBVzp1Eg4LtXdyvQpx+eXiF3XRvrSf235wlrIs5Adq6AOV5v0xpPV9DbKrHx2c0XDRLJTYGPFFFtqPxrVCZYoPcjO269HtbO3loejQjC+kwRWkvS0iX7q7pF3xYXqgPuyGGfcCiUgnnSRPhBZ1c0/ybpZUkuIDpOxD83xX3X3ZT10pfzLVJtPCMuYnN5hgMUU9uOJ/AE3+5ow='}
data_paths = {}
for name, expected in expected_files.items():
    destination = Path('data') / name
    if destination.exists():
        raw = destination.read_bytes()
    else:
        raw = zlib.decompress(base64.b64decode(saved_snapshots[name]))
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError(f'Changed project input: {name}. Review its source before continuing.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(raw)
    data_paths[name] = destination
print('Verified project data:', ', '.join(data_paths))

Look for **Verified project data**. The [data dictionary](https://github.com/ningzhanglab/Python-for-Beginners/blob/main/data/README.md) records raw API URLs, licenses, transformations and limitations. A frozen snapshot makes the exercise reproducible; it does not make the observations current.

### Inspect the complete input before selecting a sample

Load the snapshot, inspect the columns and count unavailable indicators. Each row is an economy-year. World Bank geography includes countries and territories, and source region/income aggregates have already been excluded using documented metadata.

In [ ]:
data = pd.read_csv(data_paths['internet_urban_2019.csv'])
print(data.head())
print("Input rows:", len(data))
print(data.isna().sum())
print("Duplicate economy-years:", data.duplicated(["economy_code", "year"]).sum())

All 217 economy records remain in the input. There are 28 unavailable internet-use values. The other indicator is complete. Missingness is not a zero value and may be systematically related to the question. Both indicators are for 2019; the raw source snapshot and region labels were retrieved in 2026.

### Define the analysis sample

Select complete pairs without changing the original input. Check year, unique economy keys and valid measurement ranges. Keep both percentages on the 0–100 scale; no logarithm is needed here. Explain the observations excluded, and retain the complete input for an audit.

In [ ]:
sample = data.dropna(subset=['internet_pct', 'urban_pct']).copy()
assert sample["year"].eq(2019).all()
assert not sample["economy_code"].duplicated().any()
assert sample[["internet_pct", "urban_pct"]].ge(0).all().all()
assert sample[["internet_pct", "urban_pct"]].le(100).all().all()
print("Complete analysis rows:", len(sample))
print("Regions:", sample["region"].nunique())

The analysis has 189 economies across seven source regions. All receive equal weight, so it describes economy-level associations rather than a world-population average. Complete cases need not represent economies with unavailable data. Checks can reject impossible inputs, but plausible ranges do not prove accurate measurement.

### Make the figure communicate the question

Use one point per analysis economy, label both units and 2019, and retain the selected sample. Explicit 0–100 axes keep percentage scales interpretable. Save the exact named Figure, not an unrelated current plot.

In [ ]:
#| label: fig-project-internet
#| fig-alt: "2019 scatter of 189 complete-case economies: higher urban population share is broadly associated with higher internet use, with substantial dispersion."
#| fig-cap: "World Development Indicators, 2019 observations; snapshot retrieved 2026-11-11. CC BY 4.0; ITU and UN Population Division via World Bank. Equal economy weights; descriptive association."
output_dir = Path("downloads/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
figure, axis = plt.subplots(figsize=(6, 4), layout="constrained", facecolor="white")
axis.scatter(sample["urban_pct"], sample["internet_pct"], color="#21645a", alpha=0.8, edgecolor="white", linewidth=0.4)
axis.set(xlim=(0,100), ylim=(0,100))
axis.set(xlabel='Urban population (% of total population)', ylabel='Internet users\n(% of population)', title='Internet use and urbanization · 2019')
axis.grid(axis="y", alpha=0.25)
figure.savefig(output_dir / "project-internet.png", dpi=300, facecolor="white")
plt.show()

The figure shows an association in the retained sample. Urban definitions vary across countries, and internet-use estimates vary in survey quality and coverage. This plot cannot identify the effect of moving a person to a city.

### Fit two comparable descriptive specifications

Use the same complete sample for a simple model and one adding region categories. Keep HC1 t-based approximate intervals explicit. These broad region labels come from the 2026 metadata snapshot. They are controls, not a causal identification strategy.

In [ ]:
simple = smf.ols('internet_pct ~ urban_pct', sample, missing="raise").fit(cov_type="HC1", use_t=True)
adjusted = smf.ols('internet_pct ~ urban_pct + C(region)', sample, missing="raise").fit(cov_type="HC1", use_t=True)
print("Simple coefficient:", simple.params['urban_pct'])
print("Region-adjusted coefficient:", adjusted.params['urban_pct'])
print("N:", int(simple.nobs), int(adjusted.nobs))

The simple slope is about 0.861 and the region-adjusted slope about 0.594. The unit is percentage points of internet use per percentage point of urban population share. A ten-point comparison multiplies the slope by ten, not by log(1.10). HC1 allows unequal error variances; it does not account for all spatial dependence or source-estimate uncertainty. Seven regions would be too few clusters to casually treat region clustering as a universal fix.

### Export a table with its interpretation notes

Extract results from fitted objects instead of retyping estimates. Include N, year, term and units. The selected-coefficient table intentionally omits intercept/category estimates, so the notes must state that scope.

In [ ]:
rows = []
for label, result in [("A: simple", simple), ("B: region controls", adjusted)]:
    interval = result.conf_int().loc['urban_pct']
    rows.append({"model": label, "term": 'urban_pct', "estimate": result.params['urban_pct'], "HC1_standard_error": result.bse['urban_pct'], "ci95_lower": interval.iloc[0], "ci95_upper": interval.iloc[1], "N": int(result.nobs), "year": 2019, "unit": 'internet-use percentage points per urban-share percentage point'})
results_table = pd.DataFrame(rows)
results_table.to_csv(output_dir / "project-internet.csv", index=False, float_format="%.8f")
(output_dir / "project-internet-notes.txt").write_text('2019 observations; snapshot retrieved 2026-11-11. 189 complete-case economies from 217 retained input records; equal economy weights. Selected coefficient only; outcome/predictor unit: internet-use percentage points per urban-share percentage point. HC1 standard errors and t-based approximate 95% intervals. Model B adds seven-region categories; category coefficients and intercepts are not displayed here. Region metadata are from the retrieval date. Descriptive association, not a causal effect or an individual-level relationship. Source: World Bank WDI; International Telecommunication Union (internet use) and UN Population Division (urban population). CC BY 4.0 with World Bank additional terms; no endorsement. Source estimates and definitions differ; missingness, measurement uncertainty and spatial dependence limit interpretation. See the course data dictionary for exact API requests, raw hashes and transformations.', encoding="utf-8")
print(results_table[["model", "estimate", "HC1_standard_error", "N"]].round(4))

Download the [figure](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/project-internet.png), [selected-coefficient CSV](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/project-internet.csv) and [notes](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/project-internet-notes.txt). In Colab use the Files panel. The source data have CC BY 4.0 terms and provider attribution; the course’s original presentation has no selected license. A complete research handoff should also retain the input snapshot, code, dependency file and rerun instructions.

## Try it

Work in your own copy. Add a cell for each task and attempt it before reading a solution.

### 1. Report retained observations by region, with the correct denominator.

<details><summary>Solution 1</summary>

Run this solution after your own attempt.

In [ ]:
region_summary = sample.groupby("region").agg(n_economies=("economy_code", "size"), mean_outcome=('internet_pct', "mean"))
print(region_summary.round(2))

The group counts sum to 189, the analysis sample, rather than 217 input records. These are equal-economy means of the retained sample, not population-weighted regional estimates.

</details>

### 2. Translate ten percentage points of urban share into the outcome’s units.

<details><summary>Solution 2</summary>

Run this solution after your own attempt.

In [ ]:
translated_comparison = simple.params['urban_pct'] * 10
print(translated_comparison)

The simple-model comparison is about 8.610 percentage points of internet use for ten percentage points of urban share. It is not an 8.610 percent causal increase.

</details>

### 3. Read back the export and verify its sample and selected coefficients.

<details><summary>Solution 3</summary>

Run this solution after your own attempt.

In [ ]:
reloaded = pd.read_csv(output_dir / "project-internet.csv")
print(reloaded[["model", "N", "year"]])
print(np.allclose(reloaded["estimate"], results_table["estimate"], atol=1e-8, rtol=0))

The values agree within the chosen eight-decimal export rounding. Explain why adding region changed the specification while leaving the selected sample unchanged. A smaller coefficient is not evidence that the adjusted model has removed every confounder.

</details>

## When it goes wrong

These demonstrations catch a specific intended error so the notebook continues. Unexpected errors elsewhere should still stop execution.

### A familiar column name belongs to another dataset

This input uses its own documented field names.

In [ ]:
try:
    data['internet_users']
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Inspect the actual columns and data dictionary. Renaming a variable does not establish that its units or population match another source.

### A non-unique lookup would duplicate observations

Create a deliberately duplicated key and reject a many-to-one merge that does not satisfy its contract. This changes only an in-memory practice copy.

In [ ]:
try:
    bad_lookup = pd.concat([data[["economy_code", "region"]], data[["economy_code", "region"]].iloc[[0]]])
    data[["economy_code", "year"]].merge(bad_lookup, on="economy_code", validate="many_to_one")
except pd.errors.MergeError as error:
    print(f"{type(error).__name__}: {error}")

Resolve the meaning of duplicate source keys before joining. Dropping arbitrary duplicates simply to silence this error could hide a substantive mismatch.

## Recap

- Start from documented raw provenance and retain the unfiltered input.
- Use the same explicit sample for model comparisons.
- Keep units, uncertainty assumptions and omitted coefficients visible.
- Association across economies does not establish individual or causal effects.

**Next:** [Reference pages](https://ningzhanglab.github.io/Python-for-Beginners/reference/index.html). Before sharing your own work, ask another person to rerun it and explain its sample and limits.